# Notebook 03: Hyperparameter Optimization

## Project 19 - Anomaly Detection Using Autoencoders

**Pipeline stage:** 4 of 5 - Hyperparameter Search

### Purpose

This notebook consolidates three progressively more sophisticated hyperparameter-search strategies that were explored during development, in order to identify an autoencoder configuration that improves on the baseline established in `02_baseline_autoencoder_model.ipynb`:

1. **Grid search (v1)** - manual exploration of a small, fixed set of key architectural parameters.
2. **Bayesian optimization with Optuna (v2)** - automated search using Optuna's Tree-structured Parzen Estimator (TPE) sampler.
3. **Extended search with activation tuning (v3)** - the same Optuna TPE sampler as v2, extended to also search over encoder and decoder activation functions.

Each section preserves the original experimental intent of its corresponding search strategy while calling the same shared utility functions from `src/model.py`, `src/train.py`, and `src/evaluate.py` used throughout the project, so that every candidate model is built, trained, and evaluated identically.

### Anomaly Detection Methodology Note

Every candidate configuration is evaluated using the same reconstruction-error thresholding approach used in the baseline notebook: the model is trained on normal traffic only, a threshold is calibrated on the training reconstruction-error distribution, and test samples whose reconstruction error exceeds that threshold are classified as anomalies. Hyperparameters are selected to minimize validation reconstruction loss and/or maximize the resulting F1-score.

## 1. Environment Setup

Load the required libraries, add `src/` to the import path, and configure global reproducibility seeds. `RUN_MODE` controls whether this notebook re-runs the full search (`True`) or loads previously saved results and models (`False`), so the notebook can be re-executed quickly for grading without repeating hours of search time.

In [12]:
# Standard imports
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import optuna
import joblib
from optuna.trial import TrialState
from sklearn.model_selection import train_test_split

# Project imports
src_dir = Path("../src")
sys.path.insert(0, str(src_dir))

from data_preprocessing import load_data, resolve_project_root
from evaluate import generate_anomaly_metrics_and_threshold
from model import build_autoencoder, Autoencoder
from train import set_global_seed, train_autoencoder

# Initialize project paths
project_root = resolve_project_root()
data_dir = project_root / "data" / "processed"
models_dir = project_root / "models"

# Set global seeds for reproducibility
set_global_seed(69)

# Notebook configuration
RUN_MODE = False  # Set to True to train all models, False to load a pre-trained model
N_TRIALS = 50  # Number of trials for hyperparameter optimization


def parse_saved_param_value(value: str):
    """Parse a hyperparameter value loaded from a ``best_model_params_*.txt`` file.

    Values are saved as plain ``str(value)`` text, so this recovers ints and
    floats (including scientific notation like ``1e-05``) where possible, and
    otherwise falls back to the raw string (e.g. activation names like
    ``leaky_relu``).
    """

    try:
        return int(value)
    except ValueError:
        pass
    try:
        return float(value)
    except ValueError:
        return value

## 2. Data Loading

Load the preprocessed training and test arrays produced by `01_data_preprocessing.ipynb`. These arrays are reused, unmodified, across every search strategy in this notebook so that all candidate configurations are compared on identical data.

In [13]:
# Load train and test tables
train_data_path = data_dir / "train.csv"
test_data_path = data_dir / "test.csv"

train_set = load_data(train_data_path)
test_set = load_data(test_data_path)

print("Train Set:")
display(train_set.head())
print("Test Set:")
display(test_set.head())

# Convert labels to numeric classes expected by sklearn metrics
label_to_int = {"normal": 0, "anomaly": 1}

feature_cols = [c for c in train_set.columns if c not in ["label", "binary_label"]]
X_train_df = train_set[feature_cols]
X_test_df = test_set[feature_cols]

y_train_series = train_set["binary_label"].map(label_to_int)
y_test_series = test_set["binary_label"].map(label_to_int)

if y_train_series.isna().any() or y_test_series.isna().any():
    raise ValueError("Unexpected values found in binary_label. Expected only 'normal' and 'anomaly'.")

# Default source: arrays built from CSV
X_train = X_train_df.to_numpy(dtype=np.float32)
X_test = X_test_df.to_numpy(dtype=np.float32)
y_train = y_train_series.to_numpy(dtype=np.int32)
y_test = y_test_series.to_numpy(dtype=np.int32)
source = "CSV-derived arrays"

print(f"Train Set Shape: {X_train.shape}, Labels Shape: {y_train.shape}")
print(f"Test Set Shape: {X_test.shape}, Labels Shape: {y_test.shape}")

Train Set:


,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,-0.145561,-0.026526,-0.092098,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False
1,-0.145561,-0.026472,-0.051431,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False
2,-0.145561,-0.026689,-0.086207,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False
3,-0.145561,-0.026173,-0.066131,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False
4,-0.145561,-0.025901,0.015446,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False


Test Set:


,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,-0.145561,-0.035007,-0.098556,0.0,0.0,0.0,-0.053347,-0.010067,-1.959581,-0.008538,...,False,False,False,False,True,False,False,False,False,False
1,-0.145561,-0.035007,-0.098556,0.0,0.0,0.0,-0.053347,-0.010067,-1.959581,-0.008538,...,False,False,False,False,True,False,False,False,False,False
2,-0.145561,-0.035007,-0.098556,0.0,0.0,0.0,-0.053347,-0.010067,-1.959581,-0.008538,...,True,False,False,False,False,False,False,False,False,False
3,-0.145561,-0.029163,-0.080533,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False
4,-0.145561,-0.035007,-0.098556,0.0,0.0,0.0,-0.053347,-0.010067,-1.959581,-0.008538,...,False,False,False,False,True,False,False,False,False,False


Train Set Shape: (61482, 118), Labels Shape: (61482,)
Test Set Shape: (84104, 118), Labels Shape: (84104,)


## 3. Grid Search Optimization (v1)

This section implements the original manual grid search, exhaustively evaluating a small, predefined set of hyperparameter combinations to establish initial optimization performance bounds beyond the single baseline configuration.

**Search space:**

- Latent dimension: `[8, 16, 32]`
- Learning rate: `[1e-3, 5e-4, 1e-4]`
- Batch size: `[128, 256, 512]`

**Approach:** Train and evaluate every combination in the Cartesian product of the search space above, tracking the configuration with the lowest validation reconstruction loss.

In [14]:
if RUN_MODE:
    # Define search space for baseline grid search
    latent_dims = [8, 16, 32]
    learning_rates = [1e-3, 5e-4, 1e-4]
    batch_sizes = [128, 256, 512]

    # Storage for results
    v1_results = []
    best_val_loss = float('inf')
    best_params = None
    best_model = None

    # Grid search execution
    total_combinations = len(latent_dims) * len(learning_rates) * len(batch_sizes)
    print(f"Testing {total_combinations} hyperparameter combinations...")

    combo_count = 0
    for latent_dim in latent_dims:
        for lr in learning_rates:
            for batch_size in batch_sizes:
                combo_count += 1
                print(f"\nCombination {combo_count}/{total_combinations}:")
                print(f"  Latent dim: {latent_dim}, LR: {lr}, Batch size: {batch_size}")

                # Split training data for validation
                x_fit, x_val = train_test_split(X_train, test_size=0.2, random_state=42, shuffle=True)

                # Build model
                model = build_autoencoder(
                    input_dim=X_train.shape[1],
                    latent_dim=latent_dim,
                    hidden_units=32,
                    dropout_rate=0.0,
                    n_hidden_layers=1,
                    activation_encoder='relu',
                    activation_decoder='relu'
                )

                # Compile model
                optimizer = keras.optimizers.Adam(learning_rate=lr)
                model.compile(optimizer=optimizer, loss='mse')

                # Train model
                history = model.fit(
                    x_fit, x_fit,
                    validation_data=(x_val, x_val),
                    epochs=20,
                    batch_size=batch_size,
                    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
                    verbose=0
                )

                # Record results
                val_loss = min(history.history['val_loss'])
                v1_results.append({
                    'latent_dim': latent_dim,
                    'learning_rate': lr,
                    'batch_size': batch_size,
                    'val_loss': val_loss,
                    'epochs': len(history.history['loss'])
                })

                # Track best model
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_params = {'latent_dim': latent_dim, 'learning_rate': lr, 'batch_size': batch_size}
                    best_model = model
                    print(f"  -> New best val_loss: {val_loss:.6f}")
                else:
                    print(f"  -> Val loss: {val_loss:.6f}")

    print("\n" + "="*50)
    print("BASELINE GRID SEARCH COMPLETE")
    print("="*50)
    print(f"Best validation loss: {best_val_loss:.6f}")
    print(f"Best parameters: {best_params}")
    
    # Convert results to DataFrame for analysis
    v1_df = pd.DataFrame(v1_results)
    print("\nTop 5 configurations:")
    print(v1_df.nsmallest(5, 'val_loss')[['latent_dim', 'learning_rate', 'batch_size', 'val_loss']].to_string(index=False))
    
    
    # Train the best model on the full training set with early stopping
    print("\nPreparing best model for full training...")
    best_history = best_model.fit(
        X_train, X_train,
        validation_split=0.1,
        epochs=100,
        batch_size=best_params['batch_size'],
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=0
    )
    
    # Best Model Summary
    print("\nBest Model Summary:")
    best_model.summary()
    
    # Print 
    print("\nBest Model Parameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}") 
        
        
    # Save best model parameters to a text file
    print("\nSaving best model parameters to disk...")
    with open(models_dir / "best_model_params_v1.txt", "w") as f:
        f.write("Best Model Parameters:\n")
        for key, value in best_params.items():
            f.write(f"{key}: {value}\n")    
    print("Best model parameters saved successfully.")
    
    
    # Save the best model to disk
    print("\nSaving best model to disk...")
    best_model.save(models_dir / "autoencoder_best_model_v1.keras")
    print("Best model saved successfully.")
    
    
else:
    print("Loading best model data from disk...")
    final_model = keras.models.load_model(models_dir / "autoencoder_best_model_v1.keras")

    # Load best model parameters from the text file
    best_params = {}
    with open(models_dir / "best_model_params_v1.txt", "r") as f:
        lines = f.readlines()[1:]  # Skip the first line
        for line in lines:
            key, value = line.strip().split(": ")
            best_params[key] = parse_saved_param_value(value)
    
    # Print best model parameters
    print("\nBest Model Parameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
                  
    print("\nBest Model Summary:")
    final_model.summary()

    
    

Loading best model data from disk...

Best Model Parameters:
  latent_dim: 32
  learning_rate: 0.001
  batch_size: 256

Best Model Summary:


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 118)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (Sequential)            │ (None, 32)             │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Sequential)            │ (None, 118)            │         4,950 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,444 (115.02 KB)

 Trainable params: 9,814 (38.34 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 19,630 (76.68 KB)

## 4. Bayesian Optimization with Optuna (v2)

This section replaces the manual grid search with automated hyperparameter optimization using Optuna's Tree-structured Parzen Estimator (TPE) sampler. Bayesian optimization is used here because the search space is too large to explore exhaustively, and TPE allows the search to focus on promising regions after only a modest number of trials.

**Search space:**

- Latent dimension: `[8, 16, 32, 64]`
- Learning rate: log-uniform over `[1e-4, 1e-2]`
- Batch size: `[32, 64, 128, 256, 512]`
- Dropout rate: `[0.0, 0.1, 0.2, 0.3]`
- Number of hidden layers: `[1, 2, 3]`
- Hidden units: `[16, 32, 64, 128]`

**Approach:** Optuna's TPE sampler models the relationship between hyperparameters and validation loss from previously completed trials, then proposes new trials that balance exploring under-sampled regions of the search space against exploiting configurations already known to perform well.

In [15]:
if RUN_MODE:
    # Define search space
    search_space_2 = {
        'latent_dim': [8, 16, 32, 64],
        'learning_rate': [1e-2, 1e-4], 
        'batch_size': [32, 64, 128, 256, 512],
        'dropout_rate': [0.0, 0.1, 0.2, 0.3],
        'n_hidden_layers': [1, 2, 3],
        'hidden_units': [16, 32, 64, 128], # Direct specification instead of geometric 
    }

    # Output the search space for verification
    print("\nHyperparameter Search Space:")
    for param, values in search_space_2.items():
        print(f"{param}: {values}")# 
    print("\n" + "="*50)    

    # Prepare validation split for Optuna objective
    x_fit, x_val = train_test_split(X_train, test_size=0.2, random_state=42, shuffle=True)

    # Define Optuna objective function
    def objective(trial):
        """Objective function for Optuna optimization."""
        # Sample hyperparameters
        latent_dim = trial.suggest_categorical('latent_dim', search_space_2['latent_dim'])
        learning_rate = trial.suggest_loguniform('learning_rate', search_space_2['learning_rate'][1], search_space_2['learning_rate'][0])
        batch_size = trial.suggest_categorical('batch_size', search_space_2['batch_size'])
        dropout_rate = trial.suggest_categorical('dropout_rate', search_space_2['dropout_rate'])
        n_hidden_layers = trial.suggest_categorical('n_hidden_layers', search_space_2['n_hidden_layers'])
        hidden_units = trial.suggest_categorical('hidden_units', search_space_2['hidden_units'])

        # Build model with sampled hyperparameters
        model = build_autoencoder(
            input_dim=X_train.shape[1],
            latent_dim=latent_dim,
            hidden_units=hidden_units,
            dropout_rate=dropout_rate,
            n_hidden_layers=n_hidden_layers,
            activation_encoder='relu',
            activation_decoder='relu'
        )

        # Compile model
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss='mse')

        # Train model with early stopping
        history = model.fit(
            x_fit, x_fit,
            validation_data=(x_val, x_val),
            epochs=30,
            batch_size=batch_size,
            callbacks=[
                keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
            ],
            verbose=0
        )

        # Return validation loss for minimization
        return min(history.history['val_loss'])

    # Create and run Optuna study
    print("Starting Optuna Bayesian optimization...")
    study2 = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study2.optimize(objective, n_trials=N_TRIALS, timeout=800)  # 50 trials or 15 minutes max

    # Display results
    print("\n" + "="*50)
    print("OPTUNA BAYESIAN OPTIMIZATION COMPLETE")
    print("="*50)
    print(f"Number of finished trials: {len(study2.trials)}")

    print("\nBest trial:")
    trial = study2.best_trial
    print(f"  Value (validation loss): {trial.value:.6f}")
    print("  Parameters:")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
        
        
    # Train the best model on the full training set with early stopping
    print("\nPreparing best model for full training...")
        
    # Get best parameters
    best_config = study2.best_params
    best_val_loss = study2.best_value
    
    # Train final model with best config on full training data
    print("\nTraining final model with best hyperparameters...")
    final_model = build_autoencoder(
        input_dim=X_train.shape[1],
        latent_dim=best_config['latent_dim'],
        dropout_rate=best_config['dropout_rate'],
        n_hidden_layers=best_config['n_hidden_layers'],
        hidden_units=best_config['hidden_units'],
        activation_encoder='relu',
        activation_decoder='relu'
    )

    final_model.compile(optimizer=keras.optimizers.Adam(learning_rate=best_config['learning_rate']), loss='mse')

    # Train longer on full training data with validation split
    final_history = final_model.fit(
        X_train, X_train,
        validation_split=0.1,
        epochs=100,
        batch_size=best_config['batch_size'],
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=1
    )

    # Save best model parameters to a text file
    print("\nSaving best model parameters to disk...")
    with open(models_dir / "best_model_params_v2.txt", "w") as f:
        f.write("Best Model Parameters:\n")
        for key, value in best_config.items():
            f.write(f"{key}: {value}\n")
    print("Best model parameters saved successfully.")


    # Save the final model
    final_model.save(models_dir / "autoencoder_best_model_v2.keras")
    print("Final model saved!")
    
    # Save the final study for future reference
    joblib.dump(study2, models_dir / "optuna_study_v2.pkl")
    print("Optuna study saved successfully.")
    
    # Visualize optimization history
    try:
        fig = optuna.visualization.plot_optimization_history(study2)
        fig.show()
    except Exception as e:
        print(f"Could not generate optimization history plot: {e}")

    try:
        fig = optuna.visualization.plot_param_importances(study2)
        fig.show()
    except Exception as e:
        print(f"Could not generate parameter importance plot: {e}")
else:
    # Load best model and parameters from disk
    print("Loading best model data from disk...")
    final_model = keras.models.load_model(models_dir / "autoencoder_best_model_v2.keras")

    # Load the Optuna study
    study2 = joblib.load(models_dir / "optuna_study_v2.pkl")
    print("Optuna study loaded successfully.")

    # Load best model parameters from the text file
    best_params = {}
    with open(models_dir / "best_model_params_v2.txt", "r") as f:
        lines = f.readlines()[1:]  # Skip the first line
        for line in lines:
            key, value = line.strip().split(": ")
            best_params[key] = parse_saved_param_value(value)
    
    # Print best model parameters
    print("\nBest Model Parameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
                
    print("\nBest Model Summary:")
    final_model.summary()

Loading best model data from disk...
Optuna study loaded successfully.

Best Model Parameters:
  latent_dim: 32
  learning_rate: 0.0010572705046816885
  batch_size: 256
  dropout_rate: 0.0
  n_hidden_layers: 1
  hidden_units: 32

Best Model Summary:


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 118)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (Sequential)            │ (None, 32)             │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Sequential)            │ (None, 118)            │         4,950 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,444 (115.02 KB)

 Trainable params: 9,814 (38.34 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 19,630 (76.68 KB)

## 5. Extended Search Space Exploration (v3)

This section extends the Bayesian optimization approach from Section 4 (v2) by also tuning the encoder and decoder activation functions, in addition to the architectural and optimization hyperparameters already searched in v2. Like Section 4, the search is driven by Optuna's TPE sampler; it is not a random search.

**Search space actually explored** (the `search_space_3` dictionary defined in the code cell below; a larger candidate space is kept as a commented-out reference in the code but was not the one executed for the saved results):

- Latent dimension: `[16, 32, 64]`
- Learning rate: log-uniform over `[5e-4, 1e-3]`
- Batch size: `[128, 256, 512]`
- Dropout rate: `[0.0, 0.1, 0.2]`
- Number of hidden layers: `[1, 2]`
- Hidden units: `[32, 64, 128]`
- Encoder activation: `["relu", "leaky_relu"]`
- Decoder activation: `["relu", "sigmoid", "leaky_relu"]`

**Approach:** The same trial-and-objective loop as Section 4, using the same TPE sampler and number of trials (`N_TRIALS`), but the objective function now also samples an encoder activation and a decoder activation for each trial, so this search additionally evaluates whether a non-ReLU activation improves reconstruction quality.

In [16]:
if RUN_MODE:
    # Define extended search space
    # search_space_3 = {
    #     'latent_dim': [8, 16, 32, 64, 128],
    #     'learning_rate': [1e-2, 5e-3, 1e-3, 5e-4, 1e-4],
    #     'batch_size': [32, 64, 128, 256, 512, 1024],
    #     'dropout_rate': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    #     'n_hidden_layers': [1, 2, 3, 4],
    #     'hidden_units': [16, 32, 64, 128, 256],
    #     'activation_encoder': ['relu', 'tanh', 'sigmoid','leaky_relu'],
    #     'activation_decoder': ['relu', 'sigmoid', 'linear', 'tanh','leaky_relu']
    # }

    search_space_3 = {
    'latent_dim': [16, 32, 64],
    'learning_rate': [1e-3, 5e-4, 1e-4],
    'batch_size': [128, 256, 512],
    'dropout_rate': [0.0, 0.1, 0.2],
    'n_hidden_layers': [1, 2],
    'hidden_units': [32, 64, 128],
    'activation_encoder': ['relu', 'leaky_relu'],
    'activation_decoder': ['relu', 'sigmoid','leaky_relu']
    }

# Output the search space for verification
    print("\nHyperparameter Search Space:")
    for param, values in search_space_3.items():
        print(f"{param}: {values}")# 
    print("\n" + "="*50)    

    # Prepare validation split for Optuna objective
    x_fit, x_val = train_test_split(X_train, test_size=0.2, random_state=42, shuffle=True)

    # Define Optuna objective function
    def objective(trial):
        """Objective function for Optuna optimization."""
        # Sample hyperparameters
        latent_dim = trial.suggest_categorical('latent_dim', search_space_3['latent_dim'])
        learning_rate = trial.suggest_loguniform('learning_rate', search_space_3['learning_rate'][1], search_space_3['learning_rate'][0])
        batch_size = trial.suggest_categorical('batch_size', search_space_3['batch_size'])
        dropout_rate = trial.suggest_categorical('dropout_rate', search_space_3['dropout_rate'])
        n_hidden_layers = trial.suggest_categorical('n_hidden_layers', search_space_3['n_hidden_layers'])
        hidden_units = trial.suggest_categorical('hidden_units', search_space_3['hidden_units'])
        activation_encoder = trial.suggest_categorical('activation_encoder', search_space_3['activation_encoder'])
        activation_decoder = trial.suggest_categorical('activation_decoder', search_space_3['activation_decoder'])  

        # Build model with sampled hyperparameters
        model = build_autoencoder(
            input_dim=X_train.shape[1],
            latent_dim=latent_dim,
            hidden_units=hidden_units,
            dropout_rate=dropout_rate,
            n_hidden_layers=n_hidden_layers,
            activation_encoder=activation_encoder,
            activation_decoder=activation_decoder
        )

        # Compile model
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss='mse')

        # Train model with early stopping
        history = model.fit(
            x_fit, x_fit,
            validation_data=(x_val, x_val),
            epochs=30,
            batch_size=batch_size,
            callbacks=[
                keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
            ],
            verbose=0
        )

        # Return validation loss for minimization
        return min(history.history['val_loss'])

    # Create and run Optuna study
    print("Starting Optuna Bayesian optimization...")
    study3 = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study3.optimize(objective, n_trials=N_TRIALS, timeout=800)  # 50 trials or 15 minutes max

    # Display results
    print("\n" + "="*50)
    print("OPTUNA BAYESIAN OPTIMIZATION COMPLETE")
    print("="*50)
    print(f"Number of finished trials: {len(study3.trials)}")

    print("\nBest trial:")
    trial = study3.best_trial
    print(f"  Value (validation loss): {trial.value:.6f}")
    print("  Parameters:")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
    
    # Train the best model on the full training set with early stopping
    print("\nPreparing best model for full training...")
    best_params = study3.best_params
    best_model = build_autoencoder(
        input_dim=X_train.shape[1],
        latent_dim=best_params['latent_dim'],
        hidden_units=best_params['hidden_units'],
        dropout_rate=best_params['dropout_rate'],
        n_hidden_layers=best_params['n_hidden_layers'],
        activation_encoder=best_params['activation_encoder'],
        activation_decoder=best_params['activation_decoder']
    )

    best_model.compile(optimizer=keras.optimizers.Adam(learning_rate=best_params['learning_rate']), loss='mse')

    best_history = best_model.fit(
        X_train, X_train,
        validation_split=0.1,
        epochs=100,
        batch_size=best_params['batch_size'],
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=0
    )
    
    # Best Model Summary
    print("\nBest Model Summary:")
    best_model.summary()
      
    # Save best model parameters to a text file
    print("\nSaving best model parameters to disk...")
    with open(models_dir / "best_model_params_v3.txt", "w") as f:
        f.write("Best Model Parameters:\n")
        for key, value in best_params.items():
            f.write(f"{key}: {value}\n")    
    print("Best model parameters saved successfully.")
    
    
    # Save the best model to disk
    print("\nSaving best model to disk...")
    best_model.save(models_dir / "autoencoder_best_model_v3.keras")
    print("Best model saved successfully.")
    
    # Save the final study for future reference
    print("\nSaving Optuna study to disk...")
    joblib.dump(study3, models_dir / "optuna_study_v3.pkl")
    print("Optuna study saved successfully.")
    
    # Visualize optimization history
    try:
        fig = optuna.visualization.plot_optimization_history(study3)
        fig.show()
    except Exception as e:
        print(f"Could not generate optimization history plot: {e}")

    try:
        fig = optuna.visualization.plot_param_importances(study3)
        fig.show()
    except Exception as e:
        print(f"Could not generate parameter importance plot: {e}")
        
else:
    print("Loading best model data from disk...")
    final_model = keras.models.load_model(models_dir / "autoencoder_best_model_v3.keras")

    # load the Optuna study
    study3 = joblib.load(models_dir / "optuna_study_v3.pkl")
    print("Optuna study loaded successfully.")

    # Load best model parameters from the text file
    best_params = {}
    with open(models_dir / "best_model_params_v3.txt", "r") as f:
        lines = f.readlines()[1:]  # Skip the first line
        for line in lines:
            key, value = line.strip().split(": ")
            best_params[key] = parse_saved_param_value(value)
    
    # Print best model parameters
    print("\nBest Model Parameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
                
    print("\nBest Model Summary:")
    final_model.summary()

Loading best model data from disk...
Optuna study loaded successfully.

Best Model Parameters:
  latent_dim: 64
  learning_rate: 0.0005049804820996804
  batch_size: 512
  dropout_rate: 0.0
  n_hidden_layers: 1
  hidden_units: 128
  activation_encoder: leaky_relu
  activation_decoder: leaky_relu

Best Model Summary:


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 118)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (Sequential)            │ (None, 64)             │        23,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Sequential)            │ (None, 118)            │        23,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 141,092 (551.14 KB)

 Trainable params: 47,030 (183.71 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 94,062 (367.43 KB)

## 6. Optimization Results Summary

The best configuration and validation performance from each of the three search strategies (v1, v2, v3) are compared side-by-side below. The overall best-performing model across all strategies is saved to `models/best_overall_model.keras` and is carried forward into `04_evaluation_and_visualization.ipynb` for final evaluation.

In [17]:
if RUN_MODE:
    # Build comparison data
    comparison_data = []

    # Evaluate and add baseline model if available
    baseline_model_path = models_dir / "autoencoder_baseline_model.keras"
    if baseline_model_path.exists():
        print("\nLoading baseline model for comparison...")
        baseline_model = keras.models.load_model(baseline_model_path)
        
        # Evaluate baseline model on test set
        print("Evaluating baseline model on test set...")
        baseline_threshold, baseline_y_pred, baseline_metrics = generate_anomaly_metrics_and_threshold(
            baseline_model, X_train, X_test, y_test, percentile=95
        )
        
        baseline_pred = baseline_model.predict(X_train, verbose=0)
        baseline_val_loss = np.mean(np.square(X_train - baseline_pred))
        
        comparison_data.append({
            'Approach': 'Baseline Model (Pre-trained)',
            'Best Val Loss': baseline_val_loss,
            'Latent Dim': 'N/A',  # Would need to extract from model config if needed
            'Learning Rate': 'N/A',
            'Batch Size': 'N/A',
            'Epochs': 'N/A'
        })
        print(f"Baseline Model Val Loss (proxy): {baseline_val_loss:.6f}")
        print(f"Baseline Model Test F1-Score: {baseline_metrics['f1_score']:.4f}")
        print(f"Baseline Model Test Accuracy: {baseline_metrics['accuracy']:.4f}")

    # Add baseline results if available (from v1 grid search)
    if 'v1_df' in locals() and len(v1_df) > 0:
        best_v1 = v1_df.nsmallest(1, 'val_loss').iloc[0]
        comparison_data.append({
            'Approach': 'Baseline Grid Search (v1)',
            'Best Val Loss': best_v1['val_loss'],
            'Latent Dim': best_v1['latent_dim'],
            'Learning Rate': best_v1['learning_rate'],
            'Batch Size': best_v1['batch_size'],
            'Epochs': best_v1['epochs']
        })

    # Add Optuna results if available
    if 'study2' in locals() and len(study2.trials) > 0:
        best_trial = study2.best_trial
        comparison_data.append({
            'Approach': 'Bayesian Optimization (v2)',
            'Best Val Loss': best_trial.value,
            'Latent Dim': best_trial.params.get('latent_dim', 'N/A'),
            'Learning Rate': best_trial.params.get('learning_rate', 'N/A'),
            'Batch Size': best_trial.params.get('batch_size', 'N/A'),
            'Dropout Rate': best_trial.params.get('dropout_rate', 'N/A'),
            'Hidden Layers': best_trial.params.get('n_hidden_layers', 'N/A'),
            'Hidden Units': best_trial.params.get('hidden_units', 'N/A')
        })

    # Add extended search results if available
    if 'study3' in locals() and len(study3.trials) > 0:
        best_trial = study3.best_trial
        comparison_data.append({
            'Approach': 'Extended Search (v3)',
            'Best Val Loss': best_trial.value,
            'Latent Dim': best_trial.params.get('latent_dim', 'N/A'),
            'Learning Rate': best_trial.params.get('learning_rate', 'N/A'),
            'Batch Size': best_trial.params.get('batch_size', 'N/A'),
            'Dropout Rate': best_trial.params.get('dropout_rate', 'N/A'),
            'Hidden Layers': best_trial.params.get('n_hidden_layers', 'N/A'),
            'Hidden Units': best_trial.params.get('hidden_units', 'N/A'),
            'Encoder Activation': best_trial.params.get('activation_encoder', 'N/A'),
            'Decoder Activation': best_trial.params.get('activation_decoder', 'N/A')
        })

    # Display comparison
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        print("\nOptimization Approach Comparison:")
        print("="*80)
        print(comparison_df.to_string(index=False))

        # Find the best approach (minimum validation loss)
        # Filter out rows with 'N/A' values for validation loss if needed
        valid_comparison_df = comparison_df[comparison_df['Best Val Loss'] != 'N/A'].copy()
        if len(valid_comparison_df) > 0:
            valid_comparison_df['Best Val Loss'] = pd.to_numeric(valid_comparison_df['Best Val Loss'])
            best_row = valid_comparison_df.loc[valid_comparison_df['Best Val Loss'].idxmin()]
            best_approach = best_row['Approach']
            best_val_loss = best_row['Best Val Loss']

            print(f"\nBest approach: {best_approach} with validation loss: {best_val_loss:.6f}")

            # Load the corresponding model
            if best_approach == 'Baseline Grid Search (v1)':
                model_path = models_dir / "autoencoder_best_model_v1.keras"
            elif best_approach == 'Bayesian Optimization (v2)':
                model_path = models_dir / "autoencoder_model_best_v2.keras"
            elif best_approach == 'Extended Search (v3)':
                model_path = models_dir / "autoencoder_best_model_v3.keras"
            elif best_approach == 'Baseline Model (Pre-trained)':
                model_path = models_dir / "autoencoder_baseline_model.keras"
            else:
                raise ValueError(f"Unknown approach: {best_approach}")

            print(f"Loading best model from: {model_path}")
            best_model = keras.models.load_model(model_path)

            # Evaluate on test set
            print("\nEvaluating best model on test set...")
            threshold, y_pred, metrics = generate_anomaly_metrics_and_threshold(
                best_model, X_train, X_test, y_test, percentile=95
            )
            print(f"Test Set F1-Score: {metrics['f1_score']:.4f}")
            print(f"Test Set Accuracy: {metrics['accuracy']:.4f}")

            # Save the best overall model for future use (when RUN_MODE=False)
            best_overall_path = models_dir / "best_overall_model.keras"
            best_model.save(best_overall_path)
            print(f"Best overall model saved to: {best_overall_path}")
        else:
            print("No valid optimization results available for comparison.")
    else:
        print("No optimization results available for comparison.")
else:
    try:
        # First try to load the best overall model
        best_overall_path = models_dir / "best_overall_model.keras"
        print(f"Loading best overall model from: {best_overall_path}")
        best_model = keras.models.load_model(best_overall_path)
        print("\nEvaluating best overall model on test set...")
        threshold, y_pred, metrics = generate_anomaly_metrics_and_threshold(
            best_model, X_train, X_test, y_test, percentile=95
        )
        print(f"Test Set F1-Score: {metrics['f1_score']:.4f}")
        print(f"Test Set Accuracy: {metrics['accuracy']:.4f}")
    except Exception as e:
        print(f"Could not load best overall model: {e}")
        # Fall back to evaluating the baseline model directly
        try:
            baseline_model_path = models_dir / "autoencoder_baseline_model.keras"
            print(f"Falling back to baseline model from: {baseline_model_path}")
            baseline_model = keras.models.load_model(baseline_model_path)
            print("\nEvaluating baseline model on test set...")
            threshold, y_pred, metrics = generate_anomaly_metrics_and_threshold(
                baseline_model, X_train, X_test, y_test, percentile=95
            )
            print(f"Test Set F1-Score: {metrics['f1_score']:.4f}")
            print(f"Test Set Accuracy: {metrics['accuracy']:.4f}")
        except Exception as e2:
            print(f"Could not load baseline model either: {e2}")
            print("Please run the notebook with RUN_MODE=True first to generate comparison results.")

Loading best overall model from: /Users/kallestewart/Github/SEP740-CourseProject-G9-P19-AnomalyDetection/models/best_overall_model.keras



Evaluating best overall model on test set...
Test Set F1-Score: 0.9856
Test Set Accuracy: 0.9801
